# Funciones avanzadas y análisis de datos en Python

![](src/img/logo_utb.png){width=40%}
![](src/img/logo_etd.png){width=40%}
- **Profesor:**
- **Fernando Salcedo Mejía, Eco Msc.**
- Programa de Ciencias de Datos | Escuela de transformación digital.
- 2026-1

## 1. Funciones avanzadas: lambda, map, filter y reduce

## lambda
Son funciones anónimas (sin nombre), de una sola expresión.

In [ ]:
# lambda: función anónima
cuadrado = lambda x: x**2
print(cuadrado(4))

## map
Aplica una función a cada elemento de una secuencia.

In [ ]:
# map: aplicar una función a una secuencia
numeros = [1, 2, 3, 4, 5]

# usando una función lambda para elevar al cuadrado cada número
print(list(map(lambda x: x**2, numeros)))

# invocando la función cuadrado definida anteriormente
print(list(map(cuadrado, numeros))) 

## filter
Filtra elementos que cumplen una condición lógica.

In [ ]:
# filter: filtrar elementos según condición
numeros = [1, 2, 3, 4, 5]

# usando una función lambda para filtrar números pares
print(list(filter(lambda x: x % 2 == 0, numeros)))

## reduce 
Reduce una secuencia a un único valor acumulado.

In [ ]:
from functools import reduce

numeros = [1, 2, 3, 4, 5]

# reduce: reducir una secuencia a un solo valor
print(reduce(lambda a, b: a + b, numeros))

# multiplicar todos los números de la lista usando reduce
print(reduce(lambda a, b: a * b, numeros))

## Uso de lambda y map en pandas

- lambda es la más común y recomendada dentro de pandas.
- lambda con apply() (filas o columnas)
- Se usa para : 
    - Transformaciones simples
    - Lógica condicional corta

In [ ]:
import pandas as pd
# Uso de lambda y map en pandas
datos_github = "https://raw.githubusercontent.com/fersalme/programacion-python-r/refs/heads/main/datos/palmerpenguins_extended.csv"
df_pinguinos = pd.read_csv(datos_github)

df_pinguinos


In [ ]:
# clasificar pingüinos como 'grande' o 'pequeño' según su longitud del pico
df_pinguinos['tamaño'] = df_pinguinos['bill_length_mm'].apply(lambda x: 'grande' if x > 40 else 'pequeño')
print(df_pinguinos[['bill_length_mm', 'tamaño']])

In [ ]:
# usando una funcion
def clasificar_tamaño(x):
    return 'grande' if x > 40 else 'pequeño'
df_pinguinos['tamaño_funcion'] = df_pinguinos['bill_length_mm'].apply(clasificar_tamaño)

print(df_pinguinos[['bill_length_mm', 'tamaño_funcion']])

map también se puede usar con pandas, pero es menos común que apply() para transformaciones basadas en filas o columnas. Se suele usar para aplicar funciones a Series individuales o para transformar índices.




In [ ]:
sex_encode = {'male': 'M', 'female': 'F', 'NA': 'NA'}
df_pinguinos['sex_map'] = df_pinguinos['sex'].map(sex_encode)
print(df_pinguinos[['sex', 'sex_map']])

## Tarea 1.
- Usando apply() crea una categoria de peso para los pinguinos según el cuartil de peso 25% bajo peso, 50% peso medio, 75% pesado

In [ ]:
# TU CODIGO AQUÍ

## Agrupamientos y tablas dinámicas (pivot tables)

`groupby` es una de las funciones más poderosas de Pandas. Permite aplicar la lógica :

**DataFrame -> Dividir por grupos → Aplicar función → Combinar resultado**


In [ ]:
# groupby básico
# agrupar por especie y calcular la media de la longitud del pico
media_longitud_pico = df_pinguinos.groupby('species')['bill_length_mm'].mean().reset_index()
print(media_longitud_pico)

In [ ]:
# estadisticas descriptivas media_longitud_pico por especie
estadisticas_longitud_pico = df_pinguinos.groupby('species').agg({
    'bill_length_mm': ['mean', 'median', 'std']
}).reset_index()
print(estadisticas_longitud_pico)

In [ ]:
# renombrar columnas durane el groupby
estadisticas_longitud_pico = df_pinguinos.groupby('species').agg(
    n_pinguinos=('species', 'count'),
    mean_bill_length=('bill_length_mm', 'mean'),
    median_bill_length=('bill_length_mm', 'median'),
    std_bill_length=('bill_length_mm', 'std')
).reset_index()

print(estadisticas_longitud_pico)

In [ ]:
# utilizando groupby para calcular la media de peso por especie y sexo
estadisticas_especies_sexo = df_pinguinos.groupby(['species', 'sex']).agg(
    n_pinguinos=('species', 'count'),
    mean_bill_length=('bill_length_mm', 'mean'),
    std_bill_length=('bill_length_mm', 'std'),
    mean_body_mass=('body_mass_g', 'mean'),
    std_body_mass=('body_mass_g', 'std')
).reset_index()
print(estadisticas_especies_sexo)

`pivot_table()` son tablas dinámicas para análisis multidimensional similar a las tablas dinámica de Excel

In [ ]:
# pivot table tablas de reporte calcular la media de peso por especie y sexo como tabla dinámica de Excel
pivot_especies_sexo = df_pinguinos.pivot_table(
    values='body_mass_g', # valor a agregar
    index='species', # filas
    columns='sex', # columnas
    aggfunc='mean', # función de agregación
    margins=True, # agregar totales
    margins_name="Total" # nombre para la columna de totales
)

print("Tabla dinámica de peso medio por especie y sexo:")
print(pivot_especies_sexo.round(2))

## Tarea 2.

- Crear una tabla reporte del total de pinguinos por especie y métrica de salud (health_metrics)
- Crear una tabla con la proporción de pinguinos por métrica de salud según especie

## Combinación y fusión de datasets

Pandas tiene tres operaciones principales para combinar DataFrames:

| Operación | Descripción |
|-----------|-------------|
| `pd.merge()` | Une por columna(s) clave |
| `pd.concat()` | Apila vertical u horizontalmente |
| `df.join()` | Une por el índice |


![](https://r4ds.hadley.nz/diagrams/join/venn.png)

In [ ]:
df1 = pd.DataFrame({'id': [1, 2, 3], 'nombre': ['Ana', 'Luis', 'María']})
df2 = pd.DataFrame({'id': [1, 2, 4], 'nota': [4.5, 3.8, 4.9]})

print("DataFrame 1:")
print(df1)
print("\nDataFrame 2:")
print(df2)

In [ ]:
# cruzar los DataFrames usando merge solo lo que coincide en ambos DataFrames
print("Merge (inner join) por 'id':")
df_merge = pd.merge(df1, df2, on='id', how='inner')
print(df_merge)

In [ ]:
# cruzar los DataFrames usando merge con un left join para mantener todos los registros del primer DataFrame
print("\nMerge (left join) por 'id':")
df_merge_left = pd.merge(df1, df2, on='id', how='left')
print(df_merge_left)


In [ ]:
df_q1 = pd.DataFrame({
    "mes": ["Ene","Feb","Mar"],
    "ventas": [1_200_000, 1_500_000, 1_100_000]
})
df_q2 = pd.DataFrame({
    "mes": ["Abr","May","Jun"],
    "ventas": [1_800_000, 2_100_000, 1_900_000]
})

print("\nDataFrame Q1:")
print(df_q1)
print("\nDataFrame Q2:")
print(df_q2)

In [ ]:
# Pegar los DataFrames usando concat para apilar filas
print("\nConcatenar DataFrames (apilar filas):")
df_ventas_semestre = pd.concat([df_q1, df_q2], ignore_index=True)
print(df_ventas_semestre)

## Tarea 3:

- Usando los datos de COVID-19 de NUEVOS casos confirmados (time_series_covid19_confirmed_global), NUEVAS muertes (time_series_covid19_deaths_global) y NUEVOS recuperados (time_series_covid19_recovered_global) crea un dataframen único con esta estructura:

| country_region | fecha      | casos | fallecidos | recuperados |
|----------------|------------|-------|------------|-------------|
| Colombia       | 2020-03-04 | 1     | 0          | 0           |

- Datos :
    - time_series_covid19_confirmed_global : https://raw.githubusercontent.com/fersalme/programacion-python-r/refs/heads/main/datos/time_series_covid19_confirmed_global.csv
    - time_series_covid19_recovered_global : https://raw.githubusercontent.com/fersalme/programacion-python-r/refs/heads/main/datos/time_series_covid19_recovered_global.csv
    - time_series_covid19_deaths_global : https://raw.githubusercontent.com/fersalme/programacion-python-r/refs/heads/main/datos/time_series_covid19_deaths_global.csv

- Reporta en una tabla resumen el total de casos, muertes y recuperados para Colombia.

In [ ]:
# TU CODIGO AQUÍ